In [1]:
import os
import glob
from pathlib import Path
from typing import Optional, List, Dict, Tuple

import pandas as pd


# ============================================================
# CONFIG
# ============================================================

DATA_FOLDER = r"C:\Users\TonyTang\Documents\chan.py\DataAPI\data"

INDEX_UPDATE_FOLDER = r"C:\Users\TonyTang\Documents\chan.py\DataAPI\data\index_update_month_1day_qjk5ivq"
ETF_UPDATE_FOLDER = r"C:\Users\TonyTang\Documents\chan.py\DataAPI\data\etf_update_month_1day_adjsplit_qmigryf"
ETF_5M_UPDATE_FOLDER = r"C:\Users\TonyTang\Documents\chan.py\DataAPI\data\etf_update_month_5min_adjsplit_fnso3dw"


# Existing file name -> source symbol name
# Example:
#   US10Y.csv should read update from TNX_month_1day...
#   SPY_DAY.csv should read update from SPY_month_1day_adjsplit...
SYMBOL_MAPPING = {
    "US10Y": "TNX",
    "US5Y": "FVX",
    "US30Y": "TYX",
    "QQQ_DAY": "QQQ",
    "SPY_DAY": "SPY",
    "TQQQ_DAY": "TQQQ"
}

# Prefer ETF source for these symbols
ETF_SYMBOLS = {
    "QQQ",
    "SPY",
    "TQQQ",
    # if later you add more ETF day files, put them here
}


# ============================================================
# HELPERS
# ============================================================

def normalize_existing_name(csv_stem: str) -> str:
    """
    Convert existing csv filename stem into the source symbol name.
    Examples:
        US10Y -> TNX
        SPY_DAY -> SPY
        QQQ_DAY -> QQQ
        DJI -> DJI
    """
    return SYMBOL_MAPPING.get(csv_stem, csv_stem)


def choose_source_folder(source_symbol: str) -> str:
    """
    Decide whether the source symbol should come from ETF folder or index folder.
    """
    if source_symbol in ETF_SYMBOLS:
        return ETF_UPDATE_FOLDER
    return INDEX_UPDATE_FOLDER


def find_update_file(source_symbol: str, source_folder: str) -> Optional[str]:
    """
    Find the matching update file in the source folder.

    Supports patterns like:
      AEX_month_1day.txt
      ZMUN_month_1day_adjsplit
      SPY_month_1day_adjsplit.txt
      QQQ_month_1day_adjsplit.csv
    """
    folder = Path(source_folder)

    candidates = []

    # Exact flexible patterns
    patterns = [
        f"{source_symbol}_month_1day*",
        f"{source_symbol}_month_1day_adjsplit*",
    ]

    for pattern in patterns:
        candidates.extend(folder.glob(pattern))

    # Keep files only
    candidates = [str(p) for p in candidates if Path(p).is_file()]

    # Sort by path length / name for stability
    candidates = sorted(set(candidates))

    if not candidates:
        return None

    # Prefer the more specific adjsplit file for ETF names if present
    adjsplit_candidates = [p for p in candidates if "adjsplit" in Path(p).name.lower()]
    if adjsplit_candidates:
        return adjsplit_candidates[0]

    return candidates[0]


def read_update_file(file_path: str) -> pd.DataFrame:
    """
    Read update files with either 5 columns:
        timestamp,Open,High,Low,Close
    or 6 columns:
        timestamp,Open,High,Low,Close,Volume

    QQQ/SPY adjsplit daily files include Volume. If we force only five column
    names, pandas shifts the first field into the index and the Open price is
    parsed as the timestamp, which is why source_last appeared as 1970-01-01.
    """
    raw = pd.read_csv(file_path, header=None)
    if raw.empty:
        return pd.DataFrame(columns=["timestamp", "Open", "High", "Low", "Close", "Volume"])

    if raw.shape[1] < 5:
        raise ValueError(f"Update file must have at least 5 columns: {file_path}")

    df = raw.iloc[:, :6].copy()
    if df.shape[1] == 5:
        df.columns = ["timestamp", "Open", "High", "Low", "Close"]
        df["Volume"] = 0
    else:
        df.columns = ["timestamp", "Open", "High", "Low", "Close", "Volume"]

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    for col in ["Open", "High", "Low", "Close"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["Volume"] = pd.to_numeric(df["Volume"], errors="coerce").fillna(0)

    df = df.dropna(subset=["timestamp", "Open", "High", "Low", "Close"]).copy()
    df = df.sort_values("timestamp").reset_index(drop=True)

    return df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]


def read_existing_csv(csv_path: str) -> pd.DataFrame:
    """
    Read an existing daily csv. Accepts common variants:
      timestamp,Open,High,Low,Close,Volume
      or similar case variations.
    """
    df = pd.read_csv(csv_path)

    # Find timestamp column
    ts_col = None
    for c in df.columns:
        if str(c).strip().lower() in {"timestamp", "date", "datetime", "time"}:
            ts_col = c
            break
    if ts_col is None:
        ts_col = df.columns[0]

    rename_map = {ts_col: "timestamp"}

    def find_col(candidates: List[str]) -> Optional[str]:
        lower_map = {str(c).strip().lower(): c for c in df.columns}
        for c in candidates:
            if c.lower() in lower_map:
                return lower_map[c.lower()]
        return None

    open_col = find_col(["Open", "open", "o"])
    high_col = find_col(["High", "high", "h"])
    low_col = find_col(["Low", "low", "l"])
    close_col = find_col(["Close", "close", "c", "Adj Close", "adj_close"])
    volume_col = find_col(["Volume", "volume", "vol", "v"])

    if open_col is None or high_col is None or low_col is None or close_col is None:
        raise ValueError(f"Missing OHLC columns in existing file: {csv_path}")

    rename_map[open_col] = "Open"
    rename_map[high_col] = "High"
    rename_map[low_col] = "Low"
    rename_map[close_col] = "Close"
    if volume_col is not None:
        rename_map[volume_col] = "Volume"

    df = df.rename(columns=rename_map)

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    for col in ["Open", "High", "Low", "Close"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if "Volume" not in df.columns:
        df["Volume"] = 0
    else:
        df["Volume"] = pd.to_numeric(df["Volume"], errors="coerce").fillna(0)

    df = df.dropna(subset=["timestamp", "Open", "High", "Low", "Close"]).copy()
    df = df.sort_values("timestamp").reset_index(drop=True)

    return df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]


def merge_and_update(existing_df: pd.DataFrame, update_df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
    """
    Append only rows newer than existing last timestamp.
    Then de-duplicate by timestamp and keep the latest occurrence.
    """
    if existing_df.empty:
        merged = update_df.copy()
        merged = merged.sort_values("timestamp").drop_duplicates(subset=["timestamp"], keep="last")
        return merged.reset_index(drop=True), len(merged)

    last_existing_ts = existing_df["timestamp"].max()

    # Only take rows newer than what already exists
    new_rows = update_df[update_df["timestamp"] > last_existing_ts].copy()

    if new_rows.empty:
        merged = existing_df.copy()
        merged = merged.sort_values("timestamp").drop_duplicates(subset=["timestamp"], keep="last")
        return merged.reset_index(drop=True), 0

    merged = pd.concat([existing_df, new_rows], ignore_index=True)
    merged = merged.sort_values("timestamp").drop_duplicates(subset=["timestamp"], keep="last")
    merged = merged.reset_index(drop=True)

    return merged, len(new_rows)


def save_daily_csv(df: pd.DataFrame, output_path: str) -> None:
    out = df.copy()
    out["timestamp"] = out["timestamp"].dt.strftime("%Y-%m-%d")
    out.to_csv(output_path, index=False)


def save_intraday_csv(df: pd.DataFrame, output_path: str) -> None:
    out = df.copy()
    out["timestamp"] = out["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
    out.to_csv(output_path, index=False)


def find_5m_update_file(source_symbol: str, source_folder: str = ETF_5M_UPDATE_FOLDER) -> Optional[str]:
    """
    Find a 5-minute ETF update file such as QQQ_month_5min_adjsplit.txt or SPY_month_5min_adjsplit.txt.
    """
    folder = Path(source_folder)
    patterns = [
        f"{source_symbol}_month_5min_adjsplit*",
        f"{source_symbol}_month_5min*",
        f"{source_symbol}_month_5m_adjsplit*",
        f"{source_symbol}_month_5m*",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(folder.glob(pattern))
    candidates = sorted({str(p) for p in candidates if p.is_file()})
    return candidates[0] if candidates else None


def update_one_5m_csv(existing_csv_path: str) -> None:
    existing_csv_path = str(existing_csv_path)
    csv_name = Path(existing_csv_path).name
    csv_stem = Path(existing_csv_path).stem

    source_symbol = csv_stem.replace("_5M", "").replace("_5m", "")
    update_file = find_5m_update_file(source_symbol)

    if update_file is None:
        print(f"[SKIP 5M] {csv_name} -> no 5m update file found for source symbol '{source_symbol}'")
        return

    try:
        existing_df = read_existing_csv(existing_csv_path)
        update_df = read_update_file(update_file)

        old_last = existing_df["timestamp"].max() if not existing_df.empty else None
        new_last = update_df["timestamp"].max() if not update_df.empty else None

        merged_df, added_rows = merge_and_update(existing_df, update_df)

        if added_rows == 0:
            print(
                f"[NO NEW 5M DATA] {csv_name} | source={Path(update_file).name} | "
                f"existing_last={old_last if pd.notna(old_last) else 'None'} | "
                f"source_last={new_last if pd.notna(new_last) else 'None'}"
            )
            return

        save_intraday_csv(merged_df, existing_csv_path)
        final_last = merged_df["timestamp"].max()
        print(
            f"[UPDATED 5M] {csv_name} <- {Path(update_file).name} | "
            f"added_rows={added_rows} | "
            f"old_last={old_last if pd.notna(old_last) else 'None'} | "
            f"new_last={final_last if pd.notna(final_last) else 'None'}"
        )

    except Exception as e:
        print(f"[FAILED 5M] {csv_name} | Error: {e}")


# ============================================================
# CORE UPDATE LOGIC
# ============================================================

def update_one_csv(existing_csv_path: str) -> None:
    existing_csv_path = str(existing_csv_path)
    csv_name = Path(existing_csv_path).name
    csv_stem = Path(existing_csv_path).stem

    source_symbol = normalize_existing_name(csv_stem)
    source_folder = choose_source_folder(source_symbol)
    update_file = find_update_file(source_symbol, source_folder)

    if update_file is None:
        print(f"[SKIP] {csv_name} -> no update file found for source symbol '{source_symbol}'")
        return

    try:
        existing_df = read_existing_csv(existing_csv_path)
        update_df = read_update_file(update_file)

        if existing_df.empty:
            print(f"[WARN] {csv_name} existing file is empty, will rebuild from update file only")

        old_last = existing_df["timestamp"].max() if not existing_df.empty else None
        new_last = update_df["timestamp"].max() if not update_df.empty else None

        merged_df, added_rows = merge_and_update(existing_df, update_df)

        if added_rows == 0:
            print(
                f"[NO NEW DATA] {csv_name} | source={Path(update_file).name} | "
                f"existing_last={old_last.date() if pd.notna(old_last) else 'None'} | "
                f"source_last={new_last.date() if pd.notna(new_last) else 'None'}"
            )
            return

        save_daily_csv(merged_df, existing_csv_path)

        final_last = merged_df["timestamp"].max()
        print(
            f"[UPDATED] {csv_name} <- {Path(update_file).name} | "
            f"added_rows={added_rows} | "
            f"old_last={old_last.date() if pd.notna(old_last) else 'None'} | "
            f"new_last={final_last.date() if pd.notna(final_last) else 'None'}"
        )

    except Exception as e:
        print(f"[FAILED] {csv_name} | Error: {e}")


def list_target_csvs(data_folder: str) -> List[str]:
    """
    Pick only the daily files that should be updated.
    Excludes:
      - *_5M.csv
      - raw txt files
    Includes:
      - normal daily index csvs like DJI.csv, VIX.csv
      - *_DAY.csv like SPY_DAY.csv, QQQ_DAY.csv
    """
    paths = []
    for p in Path(data_folder).glob("*.csv"):
        name = p.name.upper()

        if name.endswith("_5M.CSV"):
            continue

        # keep all other csv daily files
        paths.append(str(p))

    return sorted(paths)


def update_all_data(data_folder: str = DATA_FOLDER) -> None:
    csv_files = list_target_csvs(data_folder)

    if not csv_files:
        print(f"No csv files found in: {data_folder}")
        return

    print("=" * 90)
    print("Updating daily csv files from 30-day source update folders")
    print("=" * 90)

    for csv_path in csv_files:
        update_one_csv(csv_path)

    print("=" * 90)
    print("Done.")
    print("=" * 90)


def list_target_5m_csvs(data_folder: str) -> List[str]:
    """
    Only update the 5-minute ETF files requested by the strategy pipeline.
    """
    wanted = {"QQQ_5M.csv", "SPY_5M.csv", "TQQQ_5M.csv"}
    return sorted(str(p) for p in Path(data_folder).glob("*_5M.csv") if p.name in wanted)


def update_all_5m_data(data_folder: str = DATA_FOLDER) -> None:
    csv_files = list_target_5m_csvs(data_folder)

    if not csv_files:
        print(f"No target 5m csv files found in: {data_folder}")
        return

    print("=" * 90)
    print("Updating QQQ/SPY 5-minute csv files from 30-day source update folder")
    print("=" * 90)

    for csv_path in csv_files:
        update_one_5m_csv(csv_path)

    print("=" * 90)
    print("Done 5-minute updates.")
    print("=" * 90)


# ============================================================
# OPTIONAL: UPDATE ONLY SELECTED FILES
# ============================================================

def update_selected(symbol_names: List[str], data_folder: str = DATA_FOLDER) -> None:
    """
    symbol_names should use existing csv stem names, for example:
      ["DJI", "IXIC", "NDX", "SPY_DAY", "QQQ_DAY", "US10Y"]
    """
    for name in symbol_names:
        csv_path = os.path.join(data_folder, f"{name}.csv")
        if not os.path.exists(csv_path):
            print(f"[SKIP] missing existing csv: {csv_path}")
            continue
        update_one_csv(csv_path)


def update_selected_5m(symbol_names: List[str], data_folder: str = DATA_FOLDER) -> None:
    """
    symbol_names should use existing 5m csv stem names, for example:
      ["QQQ_5M", "SPY_5M"]
    """
    for name in symbol_names:
        csv_path = os.path.join(data_folder, f"{name}.csv")
        if not os.path.exists(csv_path):
            print(f"[SKIP 5M] missing existing csv: {csv_path}")
            continue
        update_one_5m_csv(csv_path)


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    update_all_data(DATA_FOLDER)
    update_all_5m_data(DATA_FOLDER)

    # Or only update some files:
    # update_selected([
    #     "DJI", "DXY", "FVX", "IXIC", "NDX", "RUT", "RVX", "SKEW",
    #     "SPGSCI", "US10Y", "US30Y", "US5Y", "VIX", "VIX3M", "VVIX",
    #     "VXN", "XAU", "QQQ_DAY", "SPY_DAY"
    # ])
    # update_selected_5m(["QQQ_5M", "SPY_5M"])

Updating daily csv files from 30-day source update folders
[SKIP] 10Y2YS.csv -> no update file found for source symbol '10Y2YS'
[SKIP] 10Y2YSpread.csv -> no update file found for source symbol '10Y2YSpread'
[SKIP] CPI.csv -> no update file found for source symbol 'CPI'
[UPDATED] DJI.csv <- DJI_month_1day.txt | added_rows=1 | old_last=2026-06-11 | new_last=2026-06-12
[UPDATED] DXY.csv <- DXY_month_1day.txt | added_rows=1 | old_last=2026-06-11 | new_last=2026-06-12
[UPDATED] FVX.csv <- FVX_month_1day.txt | added_rows=1 | old_last=2026-06-11 | new_last=2026-06-12
[SKIP] IXIC.csv -> no update file found for source symbol 'IXIC'
[UPDATED] NDX.csv <- NDX_month_1day.txt | added_rows=1 | old_last=2026-06-11 | new_last=2026-06-12
[SKIP] NFP.csv -> no update file found for source symbol 'NFP'
[FAILED] NYXBT.csv | Error: No columns to parse from file
[SKIP] PPI.csv -> no update file found for source symbol 'PPI'
[UPDATED] QQQ_DAY.csv <- QQQ_month_1day_adjsplit.txt | added_rows=1 | old_last=2026-0